In [0]:
# Fetch Apple stock prices for the last year using ld API
import lseg.data as ld
from datetime import datetime, timedelta

# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
%pip install lseg-data --quiet

import sys
sys.path.append('/Workspace/Users/nvaldez@tec.mx/finanzas_2026_agosto')

# Force reload of modules
import importlib
import analisis_precios_lseg.data.downloader
import analisis_precios_lseg.data.portfolio
importlib.reload(analisis_precios_lseg.data.downloader)
importlib.reload(analisis_precios_lseg.data.portfolio)

# Import after reload
from analisis_precios_lseg.data.portfolio import PortfolioDownloader

# Definir una lista de RICs para el portafolio
rics = ['AAPL.O', 'MSFT.O', 'GOOGL.O', 'AMZN.O', 'TSLA.O']

# Crear el downloader de portafolio
portfolio = PortfolioDownloader(
    rics=rics,
    app_key=APP_KEY,
    username=USERNAME,
    password=PASSWORD,
    use_secrets=False
)

# Descargar datos de los últimos 180 días
data = portfolio.download_portfolio(days=180)

# Calcular retornos diarios
returns = portfolio.get_returns(log_returns=False)


In [0]:
display(data)

In [0]:
import numpy as np
import pandas as pd

# ==============================================================================
# SECCIÓN 1: INSUMOS DEL MODELO BLACK-LITTERMAN
# ==============================================================================

# Lista de activos en el universo de inversión (N = 4)
assets = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
num_assets = len(assets)

# ------------------------------------------------------------------------------
# 1.1 Insumos del Equilibrio de Mercado (Prior)
# ------------------------------------------------------------------------------

# Pesos de Capitalización de Mercado (w_mkt): Vector N x 1 (deben sumar 1.0)
w_mkt = np.array([0.35, 0.30, 0.20, 0.15])

# Matriz de Covarianzas (Sigma): Matriz N x N (anualizada)
# Ejemplo con matriz simétrica positiva definida simulada
cov_data = [
    [0.0625, 0.0350, 0.0300, 0.0320],
    [0.0350, 0.0529, 0.0280, 0.0310],
    [0.0300, 0.0280, 0.0484, 0.0290],
    [0.0320, 0.0310, 0.0290, 0.0576]
]
Sigma = pd.DataFrame(cov_data, index=assets, columns=assets).values

# Coeficiente de Aversión al Riesgo del Mercado (Lambda / Risk Aversion)
# Típicamente entre 2.0 y 4.0 para mercados globales
lmbda = 2.5

# ------------------------------------------------------------------------------
# 1.2 Insumos de las Opiniones del Inversionista (Views)
# ------------------------------------------------------------------------------
# Supongamos K = 2 opiniones:
#   View 1 (Absoluta): AAPL tendrá un rendimiento anualizado del 12%.
#   View 2 (Relativa): MSFT superará a GOOGL por un 3% (0.03).

# Vector de retornos de las Views (Q): Vector K x 1
Q = np.array([0.12, 0.03])

# Matriz de Asignación/Dominio (P): Matriz K x N
#   Fila 1 (AAPL = 12%):   [1,  0,  0, 0]
#   Fila 2 (MSFT > GOOGL): [0,  1, -1, 0]
P = np.array([
    [1.0,  0.0,  0.0, 0.0],
    [0.0,  1.0, -1.0, 0.0]
])

# Matriz de Incertidumbre de las Views (Omega): Matriz Diagonal K x K
# Opción A (Específica): Definida manualmente según la confianza en cada view.
Omega = np.diag([0.002, 0.001])

# Opción B (Heurística de He & Litterman): Descomentar para estimar mediante P * Sigma * P^T
# tau_temp = 0.05
# Omega = np.diag(np.diag(tau_temp * P @ Sigma @ P.T))

# ------------------------------------------------------------------------------
# 1.3 Parámetro de Calibración
# ------------------------------------------------------------------------------
# Tau (tau): Escalar que pondera la incertidumbre del equilibrio (típicamente 0.01 a 0.05)
tau = 0.05

# ==============================================================================
# SECCIÓN 2: LÓGICA DE CÁLCULO Y FUNCIONES
# ==============================================================================

def calculate_implied_returns(
    lmbda: float, 
    Sigma: np.ndarray, 
    w_mkt: np.ndarray
) -> np.ndarray:
    """Calcula el vector de retornos implícitos de equilibrio (Pi) mediante Optimización Inversa."""
    return lmbda * (Sigma @ w_mkt)


def black_litterman_posterior(
    Pi: np.ndarray,
    Sigma: np.ndarray,
    P: np.ndarray,
    Q: np.ndarray,
    Omega: np.ndarray,
    tau: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula el vector de retornos a posteriori (mu_BL) y la matriz de covarianza 
    ajustada (Sigma_BL) usando Bayes.
    """
    tau_Sigma_inv = np.linalg.inv(tau * Sigma)
    Omega_inv = np.linalg.inv(Omega)

    # Matriz de precisión posterior
    M = np.linalg.inv(tau_Sigma_inv + P.T @ Omega_inv @ P)

    # Vector de retornos esperados ajustados (mu_BL)
    mu_BL = M @ (tau_Sigma_inv @ Pi + P.T @ Omega_inv @ Q)

    # Matriz de covarianza posterior ajustada por la incertidumbre de la estimación
    Sigma_BL = Sigma + M

    return mu_BL, Sigma_BL


def optimize_portfolio_weights(
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    lmbda: float
) -> np.ndarray:
    """Calcula los pesos óptimos no restringidos basados en el marco de Markowitz."""
    return np.linalg.inv(lmbda * Sigma) @ mu

# ==============================================================================
# SECCIÓN 3: EJECUCIÓN Y RESULTADOS
# ==============================================================================

# 1. Retornos Implícitos de Mercado (Prior)
Pi = calculate_implied_returns(lmbda, Sigma, w_mkt)

# 2. Actualización Bayesiana de Black-Litterman
mu_BL, Sigma_BL = black_litterman_posterior(Pi, Sigma, P, Q, Omega, tau)

# 3. Optimización para obtener pesos finales (w_BL)
w_BL = optimize_portfolio_weights(mu_BL, Sigma_BL, lmbda)


In [0]:
# Visualización comparativa de resultados
df_results = pd.DataFrame({
    'Peso Mercado (w_mkt)': w_mkt,
    'Retorno Implícito (Pi)': Pi,
    'Retorno BL (mu_BL)': mu_BL,
    'Peso Final BL (w_BL)': w_BL,
    'Diferencia (Tilt)': w_BL - w_mkt
}, index=assets)



In [0]:
# Formatear la salida del dataframe a porcentajes para facilitar lectura
df_formatted = df_results.style.format('{:.2%}')
df_formatted

In [0]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ==============================================================================
# SECCIÓN 1: INSUMOS DEL MODELO MARKOWITZ
# ==============================================================================

# Universo de activos (N = 4)
assets = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
num_assets = len(assets)

# ------------------------------------------------------------------------------
# 1.1 Insumos de Retornos Esperados (mu)
# ------------------------------------------------------------------------------
# Vector N x 1 con el rendimiento medio proyectado (anualizado) para cada activo.
# Puede provenir de promedios históricos o modelos econométricos/factores.
mu_data = [0.15, 0.12, 0.10, 0.14]
mu = np.array(mu_data)

# ------------------------------------------------------------------------------
# 1.2 Insumos de Matriz de Covarianzas (Sigma)
# ------------------------------------------------------------------------------
# Matriz N x N simétrica y positiva definida de covarianzas entre activos (anualizada).
cov_data = [
    [0.0625, 0.0350, 0.0300, 0.0320],
    [0.0350, 0.0529, 0.0280, 0.0310],
    [0.0300, 0.0280, 0.0484, 0.0290],
    [0.0320, 0.0310, 0.0290, 0.0576]
]
Sigma = pd.DataFrame(cov_data, index=assets, columns=assets).values

# ------------------------------------------------------------------------------
# 1.3 Parámetro del Entorno (Risk-Free Rate)
# ------------------------------------------------------------------------------
# Tasa libre de riesgo anualizada para el cálculo del Ratio de Sharpe
risk_free_rate = 0.04


# ==============================================================================
# SECCIÓN 2: LÓGICA DE CÁLCULO Y FUNCIONES DE OPTIMIZACIÓN
# ==============================================================================

def portfolio_performance(
    weights: np.ndarray, 
    mu: np.ndarray, 
    Sigma: np.ndarray
) -> tuple[float, float]:
    """Calcula el retorno esperado y la volatilidad (riesgo) de un portafolio."""
    portfolio_return = float(np.sum(weights * mu))
    portfolio_volatility = float(np.sqrt(weights.T @ Sigma @ weights))
    return portfolio_return, portfolio_volatility


def min_volatility_objective(weights: np.ndarray, Sigma: np.ndarray) -> float:
    """Función objetivo: Varianza del portafolio a minimizar."""
    return weights.T @ Sigma @ weights


def negative_sharpe_objective(
    weights: np.ndarray, 
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    rf: float
) -> float:
    """Función objetivo: Negativo del Sharpe Ratio (para maximizarlo)."""
    p_ret, p_vol = portfolio_performance(weights, mu, Sigma)
    return -(p_ret - rf) / p_vol


def optimize_markowitz(
    mu: np.ndarray, 
    Sigma: np.ndarray, 
    rf: float = 0.0,
    target: str = 'max_sharpe',
    allow_short: bool = False
) -> np.ndarray:
    """
    Resuelve el problema de optimización cuadrática de Markowitz.
    
    Parameters:
        mu: Vector de retornos esperados (N x 1)
        Sigma: Matriz de covarianza (N x N)
        rf: Tasa libre de riesgo
        target: 'max_sharpe' para máximo Sharpe Ratio o 'min_vol' para mínima varianza
        allow_short: Si es False, impone restricción de pesos no negativos (w >= 0)
    """
    n = len(mu)
    init_weights = np.ones(n) / n  # Pesos iniciales equi-ponderados
    
    # Restricción: La suma de pesos debe ser igual a 1.0 (Full Investment)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    
    # Limites para cada peso (Longearse únicamente vs Permitir Ventas en Corto)
    bounds = None if allow_short else tuple((0.0, 1.0) for _ in range(n))

    if target == 'max_sharpe':
        objective_fn = lambda w: negative_sharpe_objective(w, mu, Sigma, rf)
    elif target == 'min_vol':
        objective_fn = lambda w: min_volatility_objective(w, Sigma)
    else:
        raise ValueError("El parámetro target debe ser 'max_sharpe' o 'min_vol'.")

    result = minimize(
        fun=objective_fn,
        x0=init_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise RuntimeError(f"La optimización falló: {result.message}")

    return result.x


# ==============================================================================
# SECCIÓN 3: EJECUCIÓN Y RESULTADOS
# ==============================================================================

# 1. Portafolio de Máximo Sharpe Ratio (Largo únicamente: w >= 0)
w_max_sharpe = optimize_markowitz(mu, Sigma, rf=risk_free_rate, target='max_sharpe', allow_short=False)

# 2. Portafolio de Mínima Varianza Global (GMV)
w_min_vol = optimize_markowitz(mu, Sigma, rf=risk_free_rate, target='min_vol', allow_short=False)

# 3. Métricas de Rendimiento
ret_sharpe, vol_sharpe = portfolio_performance(w_max_sharpe, mu, Sigma)
ret_minvol, vol_minvol = portfolio_performance(w_min_vol, mu, Sigma)

# Tabla Comparativa de Resultados
df_results = pd.DataFrame({
    'Retorno Esperado Individual (mu)': mu,
    'Peso Max Sharpe (w*)': w_max_sharpe,
    'Peso Min Varianza (w_gmv)': w_min_vol
}, index=assets)

print("=== PORTAFOLIOS ÓPTIMOS DE MARKOWITZ ===")
print(df_results.style.format('{:.2%}').to_string())

print("\n=== PERFIL DE RIESGO Y RETORNO ===")
print(f"Max Sharpe  -> Retorno: {ret_sharpe:.2%}, Volatilidad: {vol_sharpe:.2%}, Sharpe Ratio: {(ret_sharpe - risk_free_rate)/vol_sharpe:.2f}")
print(f"Min Vol     -> Retorno: {ret_minvol:.2%}, Volatilidad: {vol_minvol:.2%}, Sharpe Ratio: {(ret_minvol - risk_free_rate)/vol_minvol:.2f}")

In [0]:
Método de selección de activos
Ventana de tiempo
Políticas de rebalanceo
Simulación de rendimientos
 